# Single qubit gate optimisation using GOAT over GRAPE

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from paraqeet.signal.pwc_generator import PWCGenerator
from paraqeet.signal.waveform import DRAGMixer

## Setup
Define Hamiltonian in the rotating frame of drive
Next, we setup the qubit system we want to control. We define the Hamiltonian in the rotating frame of drive such that the pulse oscillates slowly to apply GRAPE gradients.

The Hamiltonain in the rotating frame of the drive is given by - 
$$ H(t) = \big(\omega_q - \omega_d\big) b^\dagger b -\frac{\alpha}{2} (b^\dagger)^2 b^2 + (\epsilon(t) b + \epsilon(t)^* b)$$

In [ ]:
from paraqeet.quantity import Quantity
from paraqeet.model.closed_system import ClosedSystem
from paraqeet.model.rotating_frame_drive import RotatingFrameDrive
from paraqeet.model.transmon import Transmon


FREQ = 4e9 * 2 * np.pi
DIMS = 3
ANHARM = -200e6 * 2 * np.pi
offset = 2e6 * 2 * np.pi

drive_freq = FREQ + offset
qubit_freq = FREQ - drive_freq

First, let's generate a piecewise constant (PWC) pulse envelope for a Flattop Gaussian envelope (defined here with multiple parameters),

In [ ]:
from collections.abc import Callable
from functools import partial

from paraqeet.quantity import Array
import jax.numpy as jnp

from jax import jit
from jax.scipy.special import erf
from paraqeet.signal.envelopes import Envelope


class FlatTopGaussianEnvelope(Envelope):
    """A flat-top Gaussian envelope."""

    def __init__(self, amplitude: Quantity, t_up: Quantity, t_down: Quantity, ramp_time: Quantity, t_final: Quantity):
        self._amplitude = amplitude
        self.__t_up = t_up
        self.__t_down = t_down
        self.__ramp_time = ramp_time
        self.t_final = t_final

        self._gradient_function: Callable | None = None
        self._grad_arg_nums: tuple[int, ...] = ()

    def get_parameters(self):
        """Get all parameters of the system."""
        return [self._amplitude, self.__t_up, self.__t_down, self.__ramp_time]

    @partial(jit, static_argnums=(0,))
    def _evaluate(self, amp: Array, t_up: Array, t_down: Array, ramp_time: Array, t: Array):
        rampUp = 1 + erf((t - t_up) / ramp_time)
        rampDown = 1 + erf((-t + t_down) / ramp_time)
        return jnp.squeeze(amp * rampUp * rampDown / 4)

    def compute_output(self, t: Array) -> Array:
        """Compute pulse shape."""
        amp = self._amplitude.get_value()
        t_up = self.__t_up.get_value()
        t_down = self.__t_down.get_value()
        ramp_time = self.__ramp_time.get_value()
        return self._evaluate(amp, t_up, t_down, ramp_time, t)

In [ ]:
t_final = 25e-9
tlist = np.linspace(0, t_final, 151)

tone = FlatTopGaussianEnvelope(
    amplitude=Quantity(np.pi / t_final / 3, -np.pi / t_final, np.pi / t_final, name="Amplitude"),
    t_up=Quantity(1e-9, 0.0, t_final, name="t_up"),
    t_down=Quantity(t_final - 1e-9, 0.0, t_final, name="t_down"),
    ramp_time=Quantity(2e-9, 0.5e-9, t_final, name="ramp_time"),
    t_final=Quantity(t_final, 0.9 * t_final, 1, 1 * t_final, name="t_final"),
)

drag_tone = DRAGMixer(
    tone,
    deltas=[Quantity(2 * ANHARM, min_value=3 * ANHARM, max_value=ANHARM / 3, unit="Hz", two_pi=True, name="Delta")],
    t_final=Quantity(t_final, 0.9 * t_final, 1, 1 * t_final, name="t_final"),
)
drag_tone.multiply_flat_top = True

gen = PWCGenerator(envelopes=[drag_tone], tlist=tlist)

In [ ]:
params = drag_tone.get_parameters()
params

In [ ]:
pwc_signal = gen.get_parameters()
pwc_signal[0].set_limits(-4e9, 4e9)
pwc_signal[1].set_limits(-4e9, 4e9)

In [ ]:
ts = np.linspace(0, t_final, 1001)

plt.plot(ts / 1e-9, np.real(drag_tone.compute_output(ts)) / 1e6, label="Smooth curve")
plt.plot(ts / 1e-9, np.imag(drag_tone.compute_output(ts)) / 1e6, label="Smooth curve")
plt.plot(ts / 1e-9, np.real(gen.generate_signal(ts)) / 1e6, ls="--", label="in-phase")
plt.plot(
    ts / 1e-9,
    np.imag(gen.generate_signal(ts)) / 1e6,
    ls="--",
    label="out-of-phase",
)

plt.xlabel("Time [in ns]")
plt.ylabel("Amplitude [in MHz]")
plt.legend()

In [ ]:
Drive = RotatingFrameDrive(gen)
transmon = Transmon(
    frequency=Quantity(
        qubit_freq,
        1.2 * qubit_freq,
        0.8 * qubit_freq,
        unit="Hz",
        name="Frequency",
    ),
    anharmonicity=Quantity(ANHARM, 1.2 * ANHARM, 0.8 * ANHARM, unit="Hz", name="Anharmonicity"),
    drives=[Drive],
    dimension=DIMS,
)

model = ClosedSystem(transmon)

In [ ]:
from paraqeet.measurement.state_transfer_fidelity import StateTransferFidelityGRAPE
from paraqeet.propagation.scipy_expm_grape import ScipyExpmGRAPE


prop = ScipyExpmGRAPE(model, res=1e9)

init = np.array([[1.0], [0.0], [0]])  # |0>
target = np.array([[0.0], [1.0], [0]])  # |1>

prop.set_initial_state(init)
prop.target_state = target

prop.use_schirmer_derivative = True

zeroone = StateTransferFidelityGRAPE(
    propagation=prop,
    initial_state=init,
    target_state=target,
    times=tlist,
)

In [ ]:
def plotStates():
    """Plot the states."""
    ts = np.linspace(0, t_final, 1001)
    states = prop.propagate(ts)
    sig = gen.generate_signal(ts)

    fig, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts / 1e-9, np.real(sig), label="I")
    ax[0].plot(ts / 1e-9, np.imag(sig), label="Q")
    ax[0].legend(loc=1)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts / 1e-9, np.abs(states)[:, :, 0] ** 2)
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time [ns]")
    plt.show()


plotStates()

As expected, we get a partial transfer and a low fidelity.

In [ ]:
zeroone.measure()

We define an optimizer and link our fidelity measure as a goal function and the parameters of the cosine tone and optimise just amplitude and frequency, as in the state transfer example.

In [ ]:
drag_tone.get_parameters()

## Optimisation

In [ ]:
from paraqeet.optimisation_map import OptimisationMap
from paraqeet.optimisers.scipy_optimiser_gradient import ScipyOptimiserGradient
from paraqeet.optimisers.scipy_optimiser import ScipyOptimiser
from paraqeet.measurement.goat_over_grape import GOATOverGRAPE


optmap = OptimisationMap()
optmap.add(drag_tone)
optmap.register_params_with_optimisables()

goat = GOATOverGRAPE(zeroone, gen)

opt = ScipyOptimiser(goat, optimisables=optmap)
optgrad = ScipyOptimiserGradient(goat, optimisables=optmap)

In [ ]:
optmap

In [ ]:
optgrad.optimise()

In [ ]:
plotStates()